# Series 2.2 — Prompt Caching

**Why AI Fails? — Engineering Lab**

---

> Series 2.1 shrinks **evidence**. Series 2.2 stops **re-billing** the same stable instructions.

**Scenario:** Same HDFS investigation — but now the system prompt (role, rules, schema) is reused across hundreds of requests.

**Core lesson:** Split prompts into **static** (cacheable) and **dynamic** (per-request). Cache only what never changes.

## 1. The Problem

| Without caching | With caching (hit) |
|-----------------|-------------------|
| Static + dynamic processed every request | Static read from cache |
| Full static token cost each time | Static billed at **discounted** rate |
| ~**510** prompt tokens (v1) | ~**156** prompt tokens |
| Same answer quality | Same answer quality |

### How this differs from Series 2.1

| Lab | What it optimizes |
|-----|-------------------|
| **2.1 Context Pruning** | **How much** evidence you send (71k → 200 tokens) |
| **2.2 Prompt Caching** | **How often** you pay for the same system prompt |

Series 2.2 **builds on** 2.1 — evidence is still pruned first. Caching applies to the **static** half of the prompt.

### Why this matters in production

- Chat apps send the same system prompt on **every message**
- Agent frameworks embed rules, tool schemas, and policies on **every tool call**
- Without caching, you re-process identical text thousands of times per hour
- Savings **compound** at scale — the benchmark includes a **100-request** cost projection

## 2. Repository Layout

```
why-ai-fails/
├── common/
│   ├── prompt_builder.py      ← build_static_prompt layers + build_dynamic_prompt()
│   ├── token_usage.py         ← estimate_tokens, estimate_cost, cached input rate
│   └── gemini_client.py       ← Gemini API wrapper
├── series-2.1/
│   └── prune.py               ← evidence still pruned before dynamic prompt
└── series-2.2/
    ├── app.py                 ← CLI + benchmark runner (steps 0–8)
    ├── prompt_cache.py        ← STATIC_PROMPTS v1/v2/v3, PromptCache, billable_tokens()
    ├── benchmark.py           ← Side-by-side report + 100-request projection
    ├── README.md
    └── Series_2.2_Prompt_Caching.ipynb   ← This notebook
```

## 3. Static vs Dynamic Prompt Split

Every request is two layers:

```
┌─────────────────────────────────────┐
│  STATIC (cacheable)                 │  role, rules, schema, output format
│  build_static_prompt()              │  same every investigation
├─────────────────────────────────────┤
│  DYNAMIC (never cache)              │  user question + pruned evidence
│  build_dynamic_prompt()             │  changes every request
└─────────────────────────────────────┘
```

| Cacheable (static) | Never cache (dynamic) |
|--------------------|------------------------|
| Assistant role & persona | User question |
| Investigation rules | Filtered log evidence |
| Dataset schema | Conversation history |
| Output format template | Per-request diagnostics |

**Assembly:** `build_full_prompt(static, dynamic)` joins both before Gemini — even on cache hit, the model receives full text; savings appear in **token accounting**.

## 4. Three Layers of Caching Engineering

### Layer 1 — Split before you cache (design)

**Principle:** You cannot cache effectively until you know what is stable.

Before any cache logic runs, `app.py` builds two separate strings:

```python
static  = build_static_prompt("v1")           # ~366 tokens — CACHEABLE
dynamic = build_dynamic_prompt(question, evidence)  # ~144 tokens — NEVER CACHE
```

**Why split matters:**

- Caching the **whole** prompt would freeze user questions and log evidence — wrong answers, stale data
- Caching only the **static** block matches how Gemini, Claude, and OpenAI context caching work in production
- Series 2.1 pruning already shrunk `evidence` — dynamic layer stays small **and** fresh

**Token guard:** If you skip Layer 1 and treat everything as one blob, you either cache nothing useful or cache things that must change every request.

---

### Layer 2 — Cache the static block (the main engineering win)

**Principle:** Pay once (or at discount) for instructions that repeat on every call.

#### Cache miss vs cache hit

```
Request 1 (MISS):
  Send [full static prompt] + [dynamic question/evidence]
  → prompt_tokens = static + dynamic  (e.g. 366 + 144 = 510)

Request 2+ (HIT):
  Send [cache reference] + [dynamic question/evidence]
  → prompt_tokens = dynamic + 12 ref tokens  (e.g. 144 + 12 = 156)
  → static portion billed at CACHED_INPUT_COST_PER_M (discounted)
```

#### `billable_tokens()` — how accounting works

```python
# prompt_cache.py
CACHE_REF_TOKENS = 12   # cost to "point at" cached block

# MISS: everything fresh
return static_tokens + dynamic_tokens, 0

# HIT: dynamic fresh + small cache reference; static at discount
return dynamic_tokens + CACHE_REF_TOKENS, static_tokens
```

#### `PromptCache.resolve()` — warm-up simulation

```python
cache = PromptCache()
cache.resolve("v1", static)              # MISS — store static prompt
entry, hit = cache.resolve("v1", static)  # HIT  — read from cache
```

In production: first user pays miss cost; subsequent users get hit pricing on the static portion.

#### Cache versions (knowledge drift)

| Version | Contents | Typical tokens | Lesson |
|---------|----------|----------------|--------|
| **v1** | Lean baseline (role, rules, schema) | ~366 | Recommended starting point |
| **v2** | + security policy | ~450 | Still reasonable to cache |
| **v3** | + deprecated APIs & stale docs | ~650+ | **Cache bloat** — bigger cache = higher cost even on hit |

Run `python series-2.2/app.py --drift-demo` to see v1 → v2 → v3 growth without an API call.

> **Rule:** Cache only **stable** context. Version it. Review on a schedule — stale cached prompts cause wrong answers.

#### Production parallels

| This lab | Real production system |
|----------|------------------------|
| `STATIC_PROMPTS["v1"]` | System prompt in Gemini context cache |
| `PromptCache.resolve()` | Cache create / lookup API |
| `CACHE_REF_TOKENS` | Provider cache-read token pricing |
| v1 → v2 → v3 versions | Prompt versioning + invalidation on policy change |
| Hit ratio % | Cache hit rate dashboard (target: >90% in steady state) |

---

### Layer 3 — Measure (prove compounding savings)

**Principle:** Caching savings are invisible unless you compare billing on the **same answer**.

`app.py` makes **one** Gemini call (live mode) and builds **two** accounting rows from the same response — measuring billing differences, not answer quality.

#### What gets measured

| Metric | What it tells you |
|--------|-------------------|
| **Prompt tokens** | Input size after cache accounting |
| **Input cost** | **Primary metric caching targets** (static at full vs discounted rate) |
| **Completion tokens** | Model output — usually identical both flows |
| **Latency** | Scaled down on cache hit (fewer fresh input tokens) |
| **Cache hit ratio** | `hits / (hits + misses) × 100` |
| **100-request projection** | Cost if the same static prompt serves 100 investigations |

#### Dry-run vs live run

| Mode | Flag | API key? | What happens |
|------|------|----------|--------------|
| **Dry-run** | `--dry-run` | No | Token math via `estimate_tokens()` — **$0** |
| **Drift demo** | `--drift-demo` | No | v1/v2/v3 token sizes only — **$0** |
| **Live** | (none) | Yes | One Gemini call; both flows share the answer |

#### Example benchmark output (dry-run, v1)

```
Static tokens: 366 | Dynamic tokens: 144

WITHOUT PROMPT CACHING
Prompt Tokens     : 510
Input Cost        : $0.000038

WITH PROMPT CACHING
Prompt Tokens     : 156
Input Cost        : $0.000015
Cache Lookup      : HIT

SAVINGS (single request)
Input Cost Reduction  : ~60%
Prompt Reduction      : ~69%

SAVINGS (100 requests, cache warm)
Total Cost (no cache) : $0.0038
Total Cost (cached)   : $0.0015
Cost Reduction        : ~60%
```

#### Why input cost matters most

- **Prompt reduction %** shows token count drop (510 → 156)
- **Input cost reduction %** includes the **discounted cache rate** for static tokens
- At 100 requests/hour, a 60% input cost reduction compounds to real dollars
- Completion cost is unchanged — caching optimizes **repeated instructions**, not model output

#### Engineering checklist (present this slide)

- [ ] Split prompt into static (cacheable) and dynamic (per-request)
- [ ] Prune evidence first (Series 2.1) — keep dynamic layer small
- [ ] Version static prompts (`v1`, `v2`, `v3`) — never cache unversioned blobs
- [ ] Monitor cache hit ratio in production
- [ ] Review cached content quarterly — **knowledge drift** is a silent failure mode
- [ ] Track **input cost**, not just prompt token count

## 5. Execution Flow (`app.py` steps 0–8)

```
STEP 0  Parse CLI (--dry-run, --cache-version, --drift-demo)
    │
STEP 1  --drift-demo?  → print v1/v2/v3 token sizes → EXIT
    │
STEP 2  Load HDFS logs → prune_hdfs_context()     ← Series 2.1 still applies
    │
STEP 3  Build static prompt  (cacheable)
        Build dynamic prompt (question + evidence)
    │
STEP 4  Call Gemini once (live mode only)
    │
STEP 5  RUN 1 — without cache (cache_hit=False)
        billable_tokens(static + dynamic, cached=0)
    │
STEP 6  RUN 2 — warm cache
        resolve() → MISS, resolve() → HIT
        billable_tokens(dynamic + ref, static at discount)
    │
STEP 7  Project cost for 100 requests
    │
STEP 8  print_benchmark() — side-by-side + compounding savings
```

### Token math (typical v1 dry-run)

| Flow | Calculation | Prompt tokens |
|------|-------------|---------------|
| Without cache | 366 static + 144 dynamic | **510** |
| With cache (hit) | 144 dynamic + 12 cache ref | **156** |

## 6. How to Run

From the **repo root**:

```bash
pip install -r requirements.txt
cp .env.example .env   # optional — only for live Gemini calls
```

| Command | What it does | API key? |
|---------|--------------|----------|
| `python series-2.2/app.py --dry-run` | Token comparison, no API call | No |
| `python series-2.2/app.py --drift-demo` | v1/v2/v3 cache size comparison | No |
| `python series-2.2/app.py` | Full benchmark + one Gemini call | Yes |
| `python series-2.2/app.py --cache-version v3 --dry-run` | Cache bloat / knowledge drift demo | No |

**Custom question:**

```bash
python series-2.2/app.py --question "Investigate why HDFS block blk_-8775602795571523802 failed." --dry-run
```

In [ ]:
# Live demo — dry-run benchmark ($0, no API key)
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "series-2.2" / "app.py").exists() and (ROOT.parent / "series-2.2" / "app.py").exists():
    ROOT = ROOT.parent

result = subprocess.run(
    [sys.executable, str(ROOT / "series-2.2" / "app.py"), "--dry-run"],
    cwd=str(ROOT),
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr, file=sys.stderr)
print(f"\nExit code: {result.returncode}")

In [ ]:
# Knowledge drift demo — compare cache versions v1, v2, v3 ($0)
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "series-2.2" / "app.py").exists() and (ROOT.parent / "series-2.2" / "app.py").exists():
    ROOT = ROOT.parent

result = subprocess.run(
    [sys.executable, str(ROOT / "series-2.2" / "app.py"), "--drift-demo"],
    cwd=str(ROOT),
    capture_output=True,
    text=True,
)
print(result.stdout)

## 7. Key Code Snippets

### Billable tokens (cache accounting)

```python
# prompt_cache.py
def billable_tokens(static_tokens, dynamic_tokens, cache_hit):
    if cache_hit:
        return dynamic_tokens + CACHE_REF_TOKENS, static_tokens  # hit
    return static_tokens + dynamic_tokens, 0                     # miss
```

### Cache resolve (hit / miss)

```python
class PromptCache:
    def resolve(self, version, content):
        if version in self._store and self._store[version]["content"] == content:
            self.hits += 1
            return self._store[version], True   # HIT
        self.misses += 1
        self._store[version] = {...}
        return entry, False                     # MISS
```

### Dynamic prompt (never cache)

```python
# common/prompt_builder.py
def build_dynamic_prompt(user_question, evidence):
    return f"""User question:
{user_question}

Runtime evidence (filtered logs):
- Block ID: {evidence['block_id']}
- Error count: {evidence['error_count']}
...
Summary: {evidence['summary']}"""
```

## 8. Where Series 2.2 Fits

| Lab | Topic | What it optimizes |
|-----|-------|-------------------|
| 2.1 | Context Pruning | **Evidence size** — send less |
| **2.2** | **Prompt Caching** | **Repeated instructions** — pay less |
| 2.3 | RAG Chunking | **Retrieval quality** — right chunks |
| 2.4 | Conversation Summarization | **Chat history** — summarize, don't replay |
| 2.5 | Long-Term Memory | **Cross-session facts** — compress & store |
| 2.6 | Memory Retrieval | **100k store** — find the right memory |

**Recommended order:** Prune first (2.1), then cache the static remainder (2.2).

---

## Takeaway

> **Prune before you prompt. Cache what stays the same.**  
> Same answer. Same model. Lower input cost on every request after warm-up.

**Previous lab:** [Series 2.1 — Context Pruning](../series-2.1/) — shrink evidence before caching.

**Next lab:** [Series 2.3 — RAG Chunking](../series-2.3/) — retrieve the right evidence, not all evidence.